# Pipeline smoke test

Raw Citi Bike data -> `RawModelData` -> `ResolvedModelData` -> `Environment` -> `SimulationLog`.

The base scenario reproduces historical **demand** exactly. `FormDeparturesPhase`
re-releases every historical departure (gated by stock), and
`FormPotentialTripsPhase` assigns each departure a destination and duration from
the OD demand model `P(target | source, commodity)` + mean historical duration.

To make the replay exact, the resolved data is built with `saturate_stock=True`:
artificial saturated stock and dock capacities replace the GBFS snapshot, so the
demand gate and the overflow redirect stay in the pipeline but never bind. (The
GBFS snapshot is a current observation, unrelated to the historical start state,
and would starve the replay with stockouts that never happened.)

Because targets and durations come from the (aggregate) OD model rather than each
trip's own record, the per-trip journal is **not** identical to history, and the
OD-count matrix drifts slightly under per-period largest-remainder rounding. The
marginal that is preserved exactly is the departures table:
`simulated_departures_df == historical_departures_df`.

In [1]:
import pandas as pd

from gbp.loaders.dataloader_raw import RawModelData
from gbp.loaders.dataloader_graph import ResolvedModelData, attach_simulation
from gbp.consumers.simulator.engine import Environment, EnvironmentConfig
from gbp.consumers.simulator import (
    DockArrivals,
    FormDeparturesPhase,
    FormPotentialTripsPhase,
)

In [ ]:
# Raw model data: read the trip CSV and the live GBFS feed, derive raw entities.
raw_data = RawModelData(
    gbfs_base="https://gbfs.citibikenyc.com/gbfs/en",
    trips_path="../data/raw/202602-citibike-tripdata_1.csv",
    seed=42,
    n_depots=10,
    depot_capacity=9000,
    n_trucks=5,
    truck_capacity_bikes=20,
    truck_rate=50.0,
    electric_bike_rate=5,
    classic_bike_rate=3,
)

# Graph (resolved) model data: period grid, historical flows, replay demand.
# saturate_stock=True swaps the GBFS snapshot for artificial saturated stock and
# capacities, so demand gating and overflow redirect stay in the pipeline but
# never bind -- the base replay reproduces the historical departures exactly.
graph_data = ResolvedModelData(raw_data, period_len=pd.Timedelta(hours=1), saturate_stock=True)

historical_flows_df = graph_data.historical_flows_df

In [ ]:
phases_canonical = [
    DockArrivals("previous"),
    FormDeparturesPhase(),
    FormPotentialTripsPhase(),
    DockArrivals("same"),
]

env_canonical = Environment(
    graph_data,
    EnvironmentConfig(phases=phases_canonical, seed=42, scenario_id="historical_replay"),
)
env_canonical.run()

# Wire the finished run back into the graph-data container's simulated_* slots.
attach_simulation(graph_data, env_canonical.simulated_flows_df)
simulated_flows_df = graph_data.simulated_flows_df
simulated_departures_df = graph_data.simulated_departures_df
historical_departures_df = graph_data.historical_departures_df


def _sorted(df):
    return (df.sort_values(["period_id", "facility_id", "commodity_category"])
            .reset_index(drop=True))


# The base scenario reproduces historical demand exactly: the departures marginal
# of the simulated journal equals the historical one. (The per-trip journal is not
# identical, because targets/durations are drawn from the aggregate OD model.)
pd.testing.assert_frame_equal(_sorted(simulated_departures_df), _sorted(historical_departures_df))
print("simulated_departures_df == historical_departures_df:",
      _sorted(simulated_departures_df).equals(_sorted(historical_departures_df)))

simulated_departures_df == historical_departures_df: True


In [ ]:
# Run invariants I1-I4 (Tier-2 of the loss-logging design). validate_run is a
# pure check over the finished run: I1 demand split, I2 spine closure, I3 the
# live inventory equals the journal projection, I4 conservation. All are dormant
# in this saturated replay (no stockout, dock-full or redirect fires) and only
# bite above the baseline -- the same posture as the rest of the constraint logic.
from gbp.consumers.simulator.validation import validate_run

violations = validate_run(env_canonical.state, graph_data)

# Loss logging made the losses a filter on the journal. Under saturation both
# reasons are zero; the line proves the channel exists and the replay is loss-free.
lost = simulated_flows_df[simulated_flows_df["event_type"] == "lost"]
print("losses by reason:", lost.groupby("reason")["quantity"].sum().to_dict() or "none")
print("still in transit at horizon:", len(env_canonical.state.in_transit))

assert not violations, "run invariants violated:\n" + "\n".join(violations)
print("invariants I1-I4: OK")

In [ ]:
env_canonical

In [1]:

import pickle
dbg = pickle.load(open('D:\\Documents\\vlzm\\GFDRR\\temp\\dbg.pkl', 'rb'))

In [ ]:
import pickle
from gbp.consumers.simulator.mechanics import free_docks, dock_up_to_capacity, plan_overflow_redirect
from gbp.model import arrived_events
from gbp.consumers.simulator.state import adjust_inventory, dock_deltas

d = pickle.load(open('D:\\Documents\\vlzm\\GFDRR\\temp\\dbg.pkl', 'rb'))
state, resolved, period = d["state"], d["resolved"], d["period"]

t = period.period_id
it = state.in_transit
due = it[(it["planned_end_period"] == t) & (it["start_period"] < t)]

inventory  = state.state_inventory_df
capacities = resolved.facilities_capacities_df
occupied = inventory.groupby("facility_id")["quantity"].sum()
capacity = capacities.set_index("facility_id")["capacity"]
idx = capacity.index.union(occupied.index)
free = capacity.reindex(idx).fillna(0) - occupied.reindex(idx).fillna(0)
free = free.clip(lower=0).astype("int64")

if due.empty:
    print("no arrivals due, skipping docking and overflow redirect")
    docked, overflow = due, due
else:
    print(f"{len(due)} arrivals due, {free.sum()} free docks total")
    rank = due.groupby("planned_target_id").cumcount()
    capacity_here = due["planned_target_id"].map(free).fillna(0)
    fits = rank < capacity_here

    docked, overflow = due[fits], due[~fits]

In [5]:
capacities

,facility_id,capacity
0,4404.10,3000000
1,5666.11,3000000
2,5476.03,3000000
3,5238.05,3000000
4,6182.02,3000000
...,...,...
2280,depot_6,3000000
2281,depot_7,3000000
2282,depot_8,3000000
2283,depot_9,3000000


In [4]:
free

facility_id
1234.56    1000000
1964.01    1000000
2009.04    1000000
2042.01    1000000
2086.07    1000000
            ...   
depot_5    3000000
depot_6    3000000
depot_7    3000000
depot_8    3000000
depot_9    3000000
Length: 2285, dtype: int64

In [19]:
resolved.facilities_df

,facility_id,facility_category
0,4404.10,station
1,5666.11,station
2,5476.03,station
3,5238.05,station
4,6182.02,station
...,...,...
2280,depot_6,depot
2281,depot_7,depot
2282,depot_8,depot
2283,depot_9,depot
